**BrightGrid meters transmit every 15 minutes. Under the new streaming model, what is the realistic end-to-end latency from a fault occurrence to a dispatcher alert? What are the bottlenecks that determine that number?**

End-to-end latency could be as quick as 5-10 minutes, but definitely before a new batch of readings is generated. The bottlenecks would be the number of sources producing the readings: less sources are easier to manage and therefore it is wasier and quicker to detect faults.

_Actual bottle necks:_

  - _**File upload frequency** — the relay server batches readings and uploads every few minutes. If it uploads every 5 minutes, that's your floor regardless of how fast Spark is_.
  - _**Trigger cadence** — availableNow runs on demand. In production with a continuous trigger, the lag between file landing and Spark picking it up is the trigger interval (e.g., 30 seconds)._
  - _**Processing time** — how long the micro-batch takes to run through ingest → clean → MERGE._

**The grid operations team also runs a monthly capacity planning report that requires full historical data across all meters and all zones. Should that report use the streaming pipeline or a separate batch job? Why?**

A separate batch job connected to the same unity catalog can utilize the silver, and possibly gold, tables created from the streaming pipeline but it doesn't actually need to run repeatedly with the streaming job. Running a separate batch job either manually or on a monthly timer will suffice with gathering and aggregating the correct data when it's actually needed.

**If the streaming cluster crashes while processing a batch of readings, what mechanism prevents it from re-processing those readings when it restarts? What would happen to the current state table if MERGE ran twice on the same batch?**

I'm not sure of the exact mechanism that would prevent re-processing. I do know that in the event of a merge running twice only the unprocessed and new data would actually be processed, anything processed before the crash would be overlooked so as not duplicate any data.

_The thing preventing re-processing is the checkpoint. Auto Loader writes a record of every file it has successfully processed. On restart it reads that record and skips those files entirely — they never enter the pipeline again. The MERGE's idempotency is a safety net, not the primary guarantee. The checkpoint is the guarantee.
To be precise: MERGE would re-process the same rows if they made it through — it just wouldn't create duplicates because the MERGE condition guards against it. But the checkpoint means they don't make it through in the first place._

**BrightGrid has a handful of industrial meters that transmit every 5 minutes instead of every 15. How does this affect your watermark choice? What happens to readings from a standard meter that arrive 20 minutes late if your watermark is set to 10 minutes?**

I'm not familiar with the concept of watermarking in depth. Late readings from a standard meter would increase the latency time from a fault occurence to a dispatcher alert.

_Watermarking isn't about latency to the dispatcher — it's about memory management in Spark. Here's the actual concept:_

_Streaming aggregations (like counting events per time window) require Spark to hold state in memory — it needs to remember what it's seen so far for each window. Without a bound, Spark holds state forever for every window it's ever seen, eventually running out of memory._

_A watermark says: "I will wait at most N minutes for late data. After that, close the window and discard its state." Any event arriving after the watermark threshold for its window is simply dropped._

_For your question about a standard meter arriving 20 minutes late with a 10-minute watermark — that event gets dropped entirely. It never makes it into the clean table. That's intentional. You trade completeness for bounded memory usage. Choosing the right watermark is a business decision: how late can data realistically arrive, and what's the cost of dropping it?_

_For BrightGrid, the 5-minute industrial meters mean you need a watermark generous enough to accommodate them without dropping valid readings — 30 minutes is a reasonable choice given underground meters buffering for up to 20-30 minutes._